# Pseudonymizer - hypothesis tests

Each hypothesis = one training run with a single change against the baseline, scored on the same 500 dev docs. Baseline = `BASELINE` (a run already on Drive, e.g. `wechsel_base`).

| id | hypothesis | change |
|---|---|---|
| H1 | long context helps | `markussagen/xlm-roberta-longformer-base-4096`, window 4096 / stride 512 (full attention over 4096 - quadratic) |
| H2 | masks matter | baseline model, `--no-masks` (masked spans -> O) |
| H3 | oversampling НОМЕР matters | baseline model, `--oversample-number 0` |

Runtime: **A100**. H1 ~1 h, H2/H3 ~15 min each.

In [ ]:
#@title 1. Code
REPO = "EnderLancer/anonimimizer"  #@param {type:"string"}
BRANCH = "dmelashych/datagen"  #@param {type:"string"}
import os
# private repo: add a GitHub token as Colab secret GH_TOKEN (left sidebar -> key icon -> enable notebook access)
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if not os.path.exists('/content/ai-camp/train'):
    !rm -rf /content/ai-camp && git clone -q --branch "$BRANCH" "$url" /content/ai-camp
else:
    !cd /content/ai-camp && git pull -q
assert os.path.exists('/content/ai-camp/train'), 'clone failed: check REPO/BRANCH and the GH_TOKEN secret'
%cd /content/ai-camp/train
!pip install -q -e . 2>&1 | tail -1
import os; os.environ.update(TRANSFORMERS_VERBOSITY='error', PYTHONWARNINGS='ignore', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_PROGRESS_BARS='1')
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
print('GPU:', gpu)
if 'A100' not in gpu and 'L4' not in gpu:
    print('WARNING: not an A100 - Runtime -> Change runtime type -> A100 GPU (Colab Pro); on T4 halve every batch size')

In [ ]:
#@title 2. Data from Google Drive
from google.colab import drive
drive.mount('/content/drive')
import glob, os, shutil
from tqdm.auto import tqdm
DRIVE = '/content/drive/MyDrive/ai-camp'
os.makedirs('/content/ai-camp/data/silver', exist_ok=True); os.makedirs(f'{DRIVE}/runs', exist_ok=True)
src = f'{DRIVE}/silver' if os.path.exists(f'{DRIVE}/silver/train.parquet') else DRIVE
files = glob.glob(f'{src}/*.parquet') + glob.glob(f'{src}/*.jsonl')
assert files, f'no parquet/jsonl files found in {DRIVE} or {DRIVE}/silver'
for f in tqdm(files, desc='copy from Drive'):
    shutil.copy(f, '/content/ai-camp/data/silver/')
!ls -la /content/ai-camp/data/silver/
assert os.path.exists('/content/ai-camp/data/evaluate.py'), 'data/evaluate.py missing in the repo'
print('OK, data from', src)

In [ ]:
#@title 3. (optional) Hugging Face login - needed only for --push-to
PUSH_TO = ""  #@param {type:"string"}
if PUSH_TO:
    from google.colab import userdata
    from huggingface_hub import login
    login(userdata.get('HF_TOKEN'))  # add HF_TOKEN in Colab: left sidebar -> Secrets

In [ ]:
#@title 4. Hypotheses
BASELINE = "wechsel_base"  #@param {type:"string"}   # run name on Drive to compare against
# name: (model, extra CLI args, lr, batch, grad_accum, epochs)
HYPOTHESES = {
    'H1_longformer_4096': ('markussagen/xlm-roberta-longformer-base-4096', '--window 4096 --stride 512 --grad-checkpointing', 3e-5, 4, 8, 3),
    'H2_no_masks':        ('benjamin/roberta-base-wechsel-ukrainian', '--no-masks', 4e-5, 64, 1, 3),
    'H3_no_oversample':   ('benjamin/roberta-base-wechsel-ukrainian', '--oversample-number 0', 4e-5, 64, 1, 3),
}
RUN_ONLY = ""  #@param {type:"string"}   # e.g. H1_longformer_4096 to run a single hypothesis
SKIP_DONE = True  #@param {type:"boolean"}
N_DOCS = 0  #@param {type:"integer"}
CPU_TIMING_DOCS = 20  #@param {type:"integer"}
TODO = {k: v for k, v in HYPOTHESES.items() if not RUN_ONLY or k == RUN_ONLY}
RUNS = [BASELINE] + list(HYPOTHESES); LEADERBOARD_NAME = 'hypotheses'
import os, shutil
if os.path.exists(f'{DRIVE}/runs/{BASELINE}') and not os.path.exists(f'runs/{BASELINE}'):
    shutil.copytree(f'{DRIVE}/runs/{BASELINE}', f'runs/{BASELINE}')
print(list(TODO))

In [ ]:
#@title Train all (skips a run whose model + hyper-parameters + data were already trained; saves to Drive right after training)
import os, shutil, time, json
items = list(TODO.items()); n = len(items); t_all = time.time()
for i, (run, (model, extra, lr, batch, accum, epochs)) in enumerate(items, 1):
    pct = int(100 * (i - 1) / n)
    print(f'
[{i}/{n}  {pct}%] ===== {run}: {model} {extra} =====', flush=True)
    nd = f' --n-docs {N_DOCS}' if N_DOCS else ''
    t0 = time.time()
    # --skip-if-done: train.py compares the fingerprint with the run on Drive and exits immediately if identical
    !python scripts/train.py --model $model --out runs/$run --epochs $epochs --lr $lr --batch $batch --grad-accum $accum --dev-docs 500 --skip-if-done $DRIVE/runs/$run $extra $nd
    !rm -rf runs/$run/checkpoints && mkdir -p $DRIVE/runs/$run && cp -r runs/$run/. $DRIVE/runs/$run/
    cfg = json.load(open(f'runs/{run}/config.json')) if os.path.exists(f'runs/{run}/config.json') else {}
    print(f'[{i}/{n}] {run}: status={cfg.get("status")} best dev F1={cfg.get("best_dev_f1")} fingerprint={cfg.get("fingerprint")} '
          f'({(time.time()-t0)/60:.1f} min; total {(time.time()-t_all)/60:.0f} min) -> saved to Drive', flush=True)
print(f'
[{n}/{n}  100%] done in {(time.time()-t_all)/60:.0f} min')

In [ ]:
#@title Results vs baseline
import json, os
TYPES = ('ОСОБА','АДРЕСА','НОМЕР','ІНФОРМАЦІЯ')
def best(run):
    p = f'runs/{run}/dev_history.json'
    return max(json.load(open(p)), key=lambda h: h['final_score']) if os.path.exists(p) else None
b = best(BASELINE)
print(f"{'run':22s} {'F1':>7} {'delta':>7}  " + '  '.join(f'{t[:3]:>5}' for t in TYPES) + '   strict   risk')
for run in RUNS:
    h = best(run)
    if not h: print(f'{run:22s}  no result'); continue
    m = h['metrics']; per = m['ner_relaxed']['per_type']
    d = m['final_score'] - b['final_score'] if b else float('nan')
    print(f"{run:22s} {m['final_score']:7.4f} {d:+7.4f}  " + '  '.join(f"{per[t]['f1']:5.3f}" for t in TYPES) + f"   {m['ner_strict']['macro']['f1']:.4f}   {m['deanonymization_risk_by_name']:.3f}")

In [ ]:
#@title H1 diagnostics - do the baseline's misses sit far from the document start? (if the histogram is flat, long context cannot help)
import sys, collections
sys.path.insert(0, '../data'); import evaluate as ev
gold = {str(r['id']): r for r in ev.load_jsonl('../data/silver/dev.jsonl')}
def fn_positions(run):
    h = best(run); pred = {str(r['id']): r for r in ev.load_jsonl(f"runs/{run}/dev_submission_epoch{int(h['epoch'])}.jsonl")}
    pos = collections.Counter(); total = 0
    for k, g in gold.items():
        o = g['original']; G = ev.spans_from_annotations(g['annotations'], len(o), k); P = ev.spans_from_annotations(pred.get(k, {'annotations': []})['annotations'], len(o), k)
        mg = {i for i, _ in ev.match(G, P, False)}
        for i, s in enumerate(G):
            if i not in mg: pos[min(9, int(10 * s.start / len(o)))] += 1; total += 1
    return total, [pos[i] for i in range(10)]
for run in [r for r in RUNS if best(r)]:
    total, hist = fn_positions(run)
    print(f'{run:22s} FN total {total:4d}  by document decile: {hist}')

In [ ]:
#@title CPU inference timing (Colab CPU is slower than a laptop; use it for ranking, not absolute numbers)
import json, os, subprocess, time
from tqdm.auto import tqdm
timing = {}
if CPU_TIMING_DOCS:
    !head -n $CPU_TIMING_DOCS ../data/silver/dev.jsonl > /tmp/dev_small.jsonl
    for run in tqdm([r for r in RUNS if os.path.exists(f'runs/{r}/model')], desc='cpu timing'):
        t0 = time.time()
        subprocess.run(['python','scripts/predict.py','--model',f'runs/{run}/model','--input','/tmp/dev_small.jsonl','--output',f'/tmp/{run}_cpu.jsonl','--device','cpu','--threads','4','--batch','4'], capture_output=True)
        timing[run] = (time.time() - t0) / CPU_TIMING_DOCS
        print(f'{run:22s} {timing[run]:5.2f} s/doc -> test (240 docs) ~ {timing[run]*240/60:4.1f} min')
json.dump(timing, open(f'{DRIVE}/runs/cpu_timing_{LEADERBOARD_NAME}.json', 'w'))

In [ ]:
#@title Leaderboard (dev, official metric) -> Drive
import json, os
TYPES = ('ОСОБА','АДРЕСА','НОМЕР','ІНФОРМАЦІЯ')
def best(run):
    p = f'runs/{run}/dev_history.json'
    return max(json.load(open(p)), key=lambda h: h['final_score']) if os.path.exists(p) else None
rows = sorted(((r, best(r)) for r in RUNS), key=lambda x: -(x[1]['final_score'] if x[1] else 0))
lines = ['| run | model | best ep | macro F1 | ' + ' | '.join(t[:3] for t in TYPES) + ' | strict | risk | B3 | CPU s/doc | train min |', '|' + '---|'*12]
for run, h in rows:
    if not h: lines.append(f'| {run} | - | - | no result |'); continue
    m = h['metrics']; per = m['ner_relaxed']['per_type']; cfg = json.load(open(f'runs/{run}/config.json'))
    lines.append(f"| {run} | {cfg['model']} | {int(h['epoch'])} | **{m['final_score']:.4f}** | " + ' | '.join(f"{per[t]['f1']:.3f}" for t in TYPES) +
                 f" | {m['ner_strict']['macro']['f1']:.4f} | {m['deanonymization_risk_by_name']:.3f} | {m['consistency_b3']['f1']:.3f} | {timing.get(run, float('nan')):.2f} | {cfg['train_minutes']:.0f} |")
table = '\n'.join(lines); print(table)
open(f'{DRIVE}/runs/{LEADERBOARD_NAME}.md', 'w').write(f'# {LEADERBOARD_NAME}\n\n' + table + '\n'); print('\nsaved to', f'{DRIVE}/runs/{LEADERBOARD_NAME}.md')

In [ ]:
#@title Report (Markdown + HTML) -> Drive, shown inline
import time, shutil
from IPython.display import HTML, display
stamp = time.strftime('%Y%m%d_%H%M')
ERRORS_RUN = ""  #@param {type:"string"}   # run to dump dev errors for; empty = best run
err = f' --errors-run {ERRORS_RUN}' if ERRORS_RUN else ''
!python scripts/report.py --runs runs --out runs/report.md $err > /dev/null
for ext in ('md', 'html'):
    shutil.copy(f'runs/report.{ext}', f'{DRIVE}/runs/report_{stamp}.{ext}')
    shutil.copy(f'runs/report.{ext}', f'{DRIVE}/runs/report_latest.{ext}')
print('saved:', f'{DRIVE}/runs/report_{stamp}.md', 'and .html (also report_latest.*)')
display(HTML(open('runs/report.html', encoding='utf-8').read()))